# Chemical Search

Search compound details by compound name, SMILES, InChI, or InChIKey.

In [6]:
import os
from pprint import pprint

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.getenv("NMRXIV_BASE_URL", "https://nmrxiv.org").rstrip("/")
API_BASE = f"{BASE_URL}/api"

session = requests.Session()
session.headers.update({"Accept": "application/json", "Content-Type": "application/json"})

BASE_URL, API_BASE

('https://dev.nmrxiv.org', 'https://dev.nmrxiv.org/api')

In [7]:
def api_request(method, path, **kwargs):
    url = f"{API_BASE}{path}"
    response = session.request(method, url, timeout=30, **kwargs)
    print(f"{response.request.method} {response.url} -> {response.status_code}")
    try:
        payload = response.json()
    except ValueError:
        payload = response.text
    if not response.ok:
        pprint(payload)
        response.raise_for_status()
    return payload


def as_dataframe(payload):
    if isinstance(payload, list):
        return pd.json_normalize(payload)
    if isinstance(payload, dict):
        for key in ["data", "projects", "samples", "datasets", "items", "results"]:
            if isinstance(payload.get(key), list):
                return pd.json_normalize(payload[key])
        return pd.json_normalize(payload)
    return pd.DataFrame({"value": [payload]})


def search_compound(query, query_type):
    return api_request("POST", "/v1/search", json={"query": query, "type": query_type})

## Search chemical compounds by structure and properties​

Advanced chemical search supporting multiple query types including SMILES, InChI, InChiKey, substructure matching, similarity search, and text-based name searches. Supports filtering by molecular properties and chemical classifications. Returns paginated results with comprehensive molecular data including calculated properties, classifications, and database references.

In [8]:
result = search_compound("1S/C5H10O2/c1-4(2)3-5(6)7/h4H,3H2,1-2H3,(H,6,7)", "inchi")
result

POST https://dev.nmrxiv.org/api/v1/search -> 200


{'current_page': 1,
 'data': [{'id': 20,
   'cas': None,
   'molecular_formula': 'C5H10O2',
   'molecular_weight': 102.13,
   'smiles': None,
   'absolute_smiles': None,
   'canonical_smiles': 'CC(C)CC(=O)O',
   'inchi': None,
   'standard_inchi': 'InChI=1S/C5H10O2/c1-4(2)3-5(6)7/h4H,3H2,1-2H3,(H,6,7)',
   'inchi_key': 'GWYFCOCPABKNJV-UHFFFAOYSA-N',
   'standard_inchi_key': None,
   'sdf': '\nRDKit          2D\n\n  7  6  0  0  0  0  0  0  0  0999 V2000\n   49.3618   64.9521    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n   69.6171   64.9521    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n   79.7447   47.4105    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n   69.6171   29.8690    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n  100.0000   47.4105    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0\n   39.2342   82.4937    0.0000 O   0  0  0  0  0  0  0  0  0  0  0  0\n   39.2342   47.4105    0.0000 O   0  0  0  0  0  0  0  0  0  0  0  0\n  1  2  1  0\n  2  3  1  0\n  3  4  1  0\n  3

In [9]:
as_dataframe(result).head()

,id,cas,molecular_formula,molecular_weight,smiles,absolute_smiles,canonical_smiles,inchi,standard_inchi,inchi_key,...,has_stereo,has_variants,variants_count,workspace_samples_count,workspace_experiment_type_counts.13C NMR - 1D,workspace_experiment_type_counts.13C NMR - DEPT,workspace_experiment_type_counts.1H NMR - 1D,workspace_experiment_type_counts.1H-13C NMR - HMBC,workspace_experiment_type_counts.1H-13C NMR - HSQC,workspace_experiment_type_counts.1H-1H NMR - COSY
0,20,None,C5H10O2,102.13,None,None,CC(C)CC(=O)O,None,"InChI=1S/C5H10O2/c1-4(2)3-5(6)7/h4H,3H2,1-2H3,...",GWYFCOCPABKNJV-UHFFFAOYSA-N,...,False,False,0,1,2,1,3,1,1,1


## Try text or structure searches

Change `query_type` to values such as `text`, `SMILES`, `InChi`, or `InChiKey`.

In [ ]:
query = "citronellol"
query_type = "text"

text_results = search_compound(query, query_type)
as_dataframe(text_results).head(20)